# Webtoon Studio — Colab T4 순차 모델 실행
**Qwen3.5-9B Q4_K_M → Illustrious XL v2.0 + IP-Adapter → AI Comic Factory**.

1. 런타임을 **T4 GPU**로 선택한다.
2. Drive를 마운트한 다음 설치/모델 다운로드 셀을 실행한다(처음 1회).
3. LLM을 실행해 작품의 2컷 콘티를 저장한다.
4. LLM을 종료하고 ComfyUI 모델을 로드해 두 컷씩 작화한다.
5. 브라우저에서 /webtoon에 접속한다. 비밀 토큰은 아래 셀에서 확인한다.

이 노트북은 스크립트 실행을 위한 시작점이다. T4 실제 이미지 품질·VRAM 테스트를 통과했다는 의미는 아니다. 노출용 공개 터널은 만들지 않는다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
from pathlib import Path
repo = Path('/content/ai-comic-factory')
if not repo.is_dir():
    subprocess.run(['git', 'clone', '-b', 'feature/webtoon-studio-integration',
                    'https://gitlab.com/judjjd19/ai-comic-factory.git', str(repo)], check=True)
print('Repository:', repo)


In [ ]:
import subprocess
base = Path('/content/drive/MyDrive/WebtoonFactory')
base.mkdir(parents=True, exist_ok=True)
subprocess.run(['python', str(repo/'scripts/colab_setup.py'),
    '--project', str(repo), '--workdir', '/content',
    '--persistent', str(base), '--install', '--download-models'], check=True)


### 웹 인터페이스를 실행하기


In [ ]:
import subprocess
from pathlib import Path
log_path = Path('/content/webtoon-next.log')
with log_path.open('ab') as log:
    next_process = subprocess.Popen(
        ['npm','run','dev','--','--hostname','127.0.0.1'],
        cwd=repo, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print('Next PID:', next_process.pid, 'log:', log_path)
print('Open the Studio after Next reports ready.')


In [ ]:
from google.colab import output
# 새 탭에서 열면 Studio의 HttpOnly 세션 쿠키가 정상 작동하기 쉽다.
# Colab may warn that window mode is deprecated; use the generated link first.
output.serve_kernel_port_as_window(3000, path='/webtoon')
env_path = repo/'.env.local'
token = next(line.split('=',1)[1].strip() for line in env_path.read_text().splitlines()
             if line.startswith('WEBTOON_STUDIO_TOKEN='))
print('Studio secret (do not share):', token)


### LLM 생성 단계 — 먼저 실행
Qwen Q4 추론 서버로 **회차 콘티(2컷 단위)**를 작성하고 저장한다. `/webtoon`에서 전체 회차 기획을 수행한 뒤 이미지 단계로 넘어간다.


In [ ]:
subprocess.run(['python', str(repo/'scripts/colab_runtime.py'), 'start-llm',
    '--binary','/content/llama.cpp/build/bin/llama-server',
    '--model',str(base/'models/downloads/Qwen3.5-9B-Q4_K_M.gguf'),
    '--ctx','8192'], check=True)


### 이미지 생성 단계 — 콘티 저장 후 실행
앞선 llama-server는 자동 종료된다. ComfyUI를 로드해 **첫 컷 승인 → 둘째 컷 작화 및 승인**을 반복한다.


In [ ]:
subprocess.run(['python', str(repo/'scripts/colab_runtime.py'), 'start-image',
    '--comfy-dir','/content/ComfyUI'], check=True)
print('Return to /webtoon and render the next two panels.')


### 상태 확인 및 종료
프로세스가 중단되어도 Drive의 승인 이미지와 콘티 JSON은 남는다.


In [ ]:
subprocess.run(['python', str(repo/'scripts/colab_runtime.py'),'status'],check=True)
# Stop when needed:
# subprocess.run(['python', str(repo/'scripts/colab_runtime.py'),'stop'],check=True)
